## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### py_my_parameter 실습

(py_my_parameter/py_my_parameter.py 참고)

In [ ]:
import rclpy
from rclpy.node import Node
from rcl_interfaces.msg import ParameterType
from rcl_interfaces.msg import SetParametersResult

class ParameterNode(Node):
    def __init__(self):
        super().__init__('parameter_node')
        self.declare_parameter('my_param', 'default_value')
        param_value = self.get_parameter('my_param').get_parameter_value().string_value
        self.get_logger().info(f'Initial "my_param": {param_value}')
        self.timer = self.create_timer(2.0, self.timer_callback)
        self.add_on_set_parameters_callback(self.parameter_callback)

    def timer_callback(self):
        current_value = self.get_parameter('my_param').get_parameter_value().string_value
        self.get_logger().info(f'Current "my_param": {current_value}')
        
    def parameter_callback(self, params):
        for param in params:
            if param.name == 'my_param':
                self.get_logger().info(f'Parameter "my_param" updated to: {param.value}')
        return SetParametersResult(successful=True)
        
def main(args=None):
    rclpy.init(args=args)
    node = ParameterNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()
    
if __name__ == '__main__':
    main()

In [ ]:
from rcl_interfaces.msg import ParameterType
from rcl_interfaces.msg import SetParametersResult

내장 패키지 rcl_interfaces 의 msg 폴더에서 ParameterType(.msg)과 SetParametersResult(.msg) 파일을 가져오는 코드.

ParameterType 파일은 파라미터 타입 관련 메시지 파일이고, 이 코드에서는 실제적으로 사용되지는 않지만, 내용은 아래와 같다.

    uint8 PARAMETER_NOT_SET=0
    uint8 PARAMETER_BOOL=1
    uint8 PARAMETER_INTEGER=2
    uint8 PARAMETER_DOUBLE=3
    uint8 PARAMETER_STRING=4
    uint8 PARAMETER_BYTE_ARRAY=5
    uint8 PARAMETER_BOOL_ARRAY=6
    uint8 PARAMETER_INTEGER_ARRAY=7
    uint8 PARAMETER_DOUBLE_ARRAY=8
    uint8 PARAMETER_STRING_ARRAY=9

SetParametersResult은 파라미터 변경 성공 여부를 반환할 때 사용하는 타입이며, 

callback 마지막에서 사용되고, 내용은 아래와 같다.

    bool successful
    string reason

In [ ]:
class ParameterNode(Node):
    def __init__(self):
        super().__init__('parameter_node')
        self.declare_parameter('my_param', 'default_value')
        param_value = self.get_parameter('my_param').get_parameter_value().string_value
        self.get_logger().info(f'Initial "my_param": {param_value}')
        self.timer = self.create_timer(2.0, self.timer_callback)
        self.add_on_set_parameters_callback(self.parameter_callback)

`self.declare_parameter('my_param', 'default_value')` : 이 노드에 `my_param`이라는 파라미터 생성. 그 값은 `default_value`로 설정한다.

값은 넣는 자리('default_value')에 어떤 값을 넣었는지에 따라 ROS2가 스스로 판단하여 타입을 결정짓는다.

예를 들어 10을 넣으면 int, 3.14를 넣으면 Double, 지금처럼 ''로 넣으면 String, True 또는 False를 넣으면 bool 타입이 된다.

`self.get_parameter('my_param')` : name이 'my_param'인 파라미터 객체 가져오기.

`get_parameter_value()` : 실제 값 객체 가져오기.

`string_value` : 그 객체의 string 값 가져오기.

`param_value = self.get_parameter('my_param').get_parameter_value().string_value` : 그 값을 `param_value`에 저장.

`self.timer = self.create_timer(2.0, self.timer_callback)` : 2초마다 `self.timer_callback` 함수를 실행하는 타이머 생성.

`self.add_on_set_parameters_callback(self.parameter_callback)` : `parameter_callback` 함수를 ROS2 시스템에 등록한다.

다른 터미널 등에서 파라미터 변경 요청이 들어오면 `self.parameter_callback` 함수를 실행한다.

In [ ]:
def timer_callback(self):
    current_value = self.get_parameter('my_param').get_parameter_value().string_value
    self.get_logger().info(f'Current "my_param": {current_value}')

파라미터의 값을 받고 주기적으로(2초마다) 터미널에 출력하는 함수.

In [ ]:
def parameter_callback(self, params):
    for param in params:
        if param.name == 'my_param':
            self.get_logger().info(f'Parameter "my_param" updated to: {param.value}')
    return SetParametersResult(successful=True)

기본적으로 파라미터 변경 요청은 1개 또는 여러개로 들어 올 수 있으며, 주로 아래 구조처럼 온다.

```python
params = [
    Parameter(name='my_param', value='hello'),          # name, value 형태
    Parameter(name='speed', value=10),                  # name, value 형태
    Parameter(name='kp', value=0.1)                     # name, value 형태
]
```
그래서 반복문으로 먼저 파라미터를 하나씩 살피고, 만약 name이 'my_param'인 파라미터 변경 요청이 있다면 터미널에 안내문을 출력하는 코드이다.

또한 return문의 `SetParametersResult(successful=True)` 객체는 아래와 같은 구조로 되어있는데

```python
SetParametersResult(
    successful=True,                # successfult = True 옵션이 반영된 결과이며, 만약 변경 거부를 하고싶다면 이 값을 False로 바꾸면 된다.
    reason='speed must be positive'
)
```

`successful = True` 옵션으로 주었기 때문에 변경 요청 수락 응답을 ROS2 시스템에 보내며, 

이때는 `reason`이 파라미터 변경을 요청한 터미널에 뜨진 않지만,

만약 `successful = False` 옵션으로 변경 요청 거부 응답을 ROS2 시스템에 보내는 경우

`reason`의 내용이 파라미터 변경을 요청한 터미널에 뜨게 된다.

참고로 터미널에서 파라미터 변경 요청을 보내는 명령어는 아래처럼 작성한다.

```bash
# 1개만 요청하는 경우
ros2 param set /parameter_node speed 10         # 노드이름, 파라미터 이름, 값 구조.
```

```bash
# 여러개 요청을 하기 위해 파일로 묶어서 요청하는 경우
ros2 param load /parameter_node params.yaml     # 노드이름, 파일 구조.
```

(params.yaml 내용은 아래처럼 작성한다.)

```yaml
/parameter_node:                # 노드 이름
  ros__parameters:              # 이건 고정이름. 바꾸면 안됨.
    speed: 10                   # 변수이름 1 : 변수값 1
    kp: 0.1                     # 변수이름 2 : 변수값 2
    mode: auto                  # 변수이름 3 : 변수값 3 (yaml에서는 ''없이도 string으로 들어간다.)
```

\#\# 참고

위 코드는 반복문의 밖에 return문이 나와있는 구조이기 때문에 사실상 무조건 변경요청 수락하는 코드이다.

main() 함수 내용은 구조가 동일하므로 생략한다.

\#\# 파라미터 노드에서 많이 쓰이는 함수

##### 1. declare_parameters()

여러 개 한 번에 선언

```python
self.declare_parameters(
    namespace='',
    parameters=[
        ('speed', 10),
        ('kp', 0.1),
        ('mode', 'auto')
    ]
)
```

\#\# 참고 

여기서는 namespace = '' 로 비어있지만

이 namespace는 파라미터 name을 만들 때 'car.speed', 'car.kp', 'car.mode'처럼 앞에 붙는 접두어 prefix를 의미하며,

이렇게 명명된 파라미터들은 실제로 호출할 때도 접두어가 붙은 형태인 'car.speed' ,'car.kp', 'car.mode' 로 호출해야 한다.

파라미터가 많아질 때 구분하기 위해 사용된다.

##### 2. get_parameters()

여러 개 읽기

```python
params = self.get_parameters([
    'speed',
    'kp',
    'mode'
])
```

##### 3. set_parameters()

여러 파라미터 수정하기 (1개도 1개만 쓰면 가능)

```python
self.set_parameters([
    Parameter('speed', Parameter.Type.INTEGER, 20)
])
```

이 경우도 `callback`함수가 호출된다.

##### 4. undeclare_parameter()

파라미터 제거

```python
self.undeclare_parameter('speed')
```

##### 5. has_parameter()

존재 여부 확인

```python
self.has_parameter('speed')
```

True 또는 False 리턴.

##### 6. list_parameters()

목록 조회

```python
result = self.list_parameters([], depth=10)
print(result.names, result.prefixes)
```

##### 7. describe_parameter()

정보 조회

```python
info = self.describe_parameter('speed')
```

#### 3. setup.py 내용추가

setup.py의 entry_points 안의 'console_scripts'에 아래와 같이 내용을 추가하여 ros2 명령어를 사용할 수 있게 만든다.

In [ ]:
entry_points={
    'console_scripts': [
        'parameter_node = py_my_parameter.parameter_node:main',
    ],
}

xml 수정과정은 의존하는 추가 패키지가 없으므로 생략.

#### 4. 빌드

```bash
cd ~/ros2_ws
```

워크스페이스로 돌아와서

```bash
colcon build
```

빌드를 해준다.(--symlink-install 옵션 넣어도 된다.(권장))

#### 5. 실행 (py_my_parameter 패키지 실습)

터미널 2개를 띄워서

양쪽 모두 아래 명령어를 실행하고

```bash
source install/setup.bash
```

한쪽에

```bash
ros2 run py_my_parameter parameter_node
```

를 실행한다.

"my_param"의 내용 'default_value'가 잘 나오는지 확인한다.

이후 반대쪽 터미널에 

```bash
ros2 param set /parameter_node my_param "new_value"
```

를 실행한다.

파라미터를 출력하던 터미널에서

변경된 파라미터 값이 잘 나오는지 확인한다.

#### 6. launch파일 실습

(py_my_parameter/launch/parameter.launch.py 참고.)

이번엔 아래 트리구조처럼 py_my_parameter 패키지 폴더 아래 launch 폴더를 만들고 

그 안에 `parameter.launch.py`파일을 작성한다.
```txt
├── launch
│   └── parameter.launch.py
├── package.xml
├── py_my_parameter
│   ├── __init__.py
│   └── parameter_node.py
├── resource
│   └── py_my_parameter
├── setup.cfg
├── setup.py
└── test
    ├── test_copyright.py
    ├── test_flake8.py
    └── test_pep257.py
```

#### parameter.launch.py 코드

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    return LaunchDescription([
        Node(
            package='py_my_parameter',
            executable='parameter_node',
            name='parameter_node',
            parameters=[{
                'my_param': 'hello_param'
            }]
        )
    ])

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

`from launch import LaunchDescription` : 기본 내장 launch 패키지에서 LaunchDescription 클래스를 가져오는 코드.

LaunchDescription은 launch 파일의 전체 실행 계획을 담는 객체로, 아래와 같은 구조라면

```python
LaunchDescription([
    Node(...),
    Node(...)
])
```

이 의미는 이 launch 파일을 실행하면 Node 1,Node 2, Node 3, ... 를 순차적으로 실행하라는 의미이다.

`from launch_ros.actions import Node` : 기본 내장 launch_ros 패키지에서 ROS2 노드 실행용 Node 클래스를 가져오는 코드.

rclpy.node 와 같은 기능을 한다고 보면 된다.

In [ ]:
Node(
    package='py_my_parameter',
    executable='parameter_node',
    name='parameter_node',
    parameters=[{
        'my_param': 'hello_param'
    }]
)

`py_my_parameter` 패키지의 `parameter_node` 실행파일(parameter_node.py)을

`parameter_node`라는 이름으로 실행하고,

`my_param` 파라미터의 값을 'hello_param'으로 설정하라는 의미.

\#\# 참고 

`executable='parameter_node'`에서의 `parameter_node`는 

setup.py의 내용 `parameter_node = py_my_parameter.parameter_node:main` 에서

앞쪽 parameter_node를 작성해야 한다. (:main 앞의 parameter_node를 쓰는 것이 아님.)

즉 예를들어 setup.py에 `node_alias = pkg.program:main` 이렇게 되어있으면

`executable = node_alias`로 작성하면 된다.

또한 name은 노드의 실제 이름이므로 원하는 대로 작성하면 된다.

가령 `name = param_launch`로 작성했다면

이후 이 런치파일을 ros2 명령어로 실행한 후 ros2 node list로 노드 이름을 검색했을 때 `/param_launch`가 뜬다.

#### 7. 실행 (py_my_parameter launch 파일 실습)

터미널 2개를 띄워서

양쪽 모두 아래 명령어를 실행하고

```bash
source install/setup.bash
```

한쪽에

```bash
ros2 launch py_my_parameter py_my_parameter.launch.py
```

를 실행한다.

"my_param"의 내용 'default_value'가 잘 나오는지 확인한다.

이후 반대쪽 터미널에 

```bash
ros2 param set /parameter_node my_param "new_value"
```

를 실행한다.

파라미터를 출력하던 터미널에서

변경된 파라미터 값이 잘 나오는지 확인한다.